# Phase 6.6 — downside and trend feature comparison

This notebook tests five related inputs as one group: one-day return, downside deviation, drawdown from a recent peak, and price compared with its 20- and 50-session averages. It compares all 14 approved market inputs with exactly the same set **without this downside-and-trend group**.

It fits models. Locally, select the project `.venv` kernel. In Colab, the setup cell clones the repository, downloads hash-verified release inputs, and installs the pinned dependencies. It creates a temporary evidence folder and refuses to overwrite an existing one. Download the evidence before ending the Colab runtime.

In [ ]:
from pathlib import Path
import subprocess
import sys

BRANCH = 'feature/model-evaluation'
COLAB_REPOSITORY = Path('/content/HalalifyAI')
if 'google.colab' in sys.modules:
    if not (COLAB_REPOSITORY / '.git').is_dir():
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/frager098/HalalifyAI.git', str(COLAB_REPOSITORY)], check=True)
    root = COLAB_REPOSITORY.resolve()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt'), '-r', str(root / 'requirements-cleaning.txt'), 'gdown>=5.2,<6'], check=True)
else:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / 'src' / 'model_training').is_dir():
        root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.notebook_environment import ensure_phase5_dataset, prepare_repository
root = prepare_repository(branch=BRANCH, colab_repository=COLAB_REPOSITORY)
ensure_phase5_dataset(root)
%cd {root}
print(f'Project folder: {root}')

## Run the controlled comparison

The next cell uses only the exact dataset and chronological protocol continued from Phase 5. It tunes only on earlier inner-validation data, then scores each later outer fold once. Expect a few minutes of work.

In [ ]:
from src.model_training.feature_ablation.phase5_continuation import (
    DOWNSIDE_TREND_OUTPUT,
    prepare_inputs,
    write_downside_trend_report,
)

prepare_inputs()
report = write_downside_trend_report()
print('Status:', report['status'])
print('Evidence folder:', DOWNSIDE_TREND_OUTPUT)
for task, summary in report['task_summaries'].items():
    print(f"{task}: {summary['decision']} | {summary['primary_metric']} difference = {summary['mean_difference']:.4f}")

## Read the result

`retain` means this related group consistently helped. `reject` means it did not help in most periods. `inconclusive` means the evidence is mixed. For risk, High-risk detection must not become worse and the number of High-risk outcomes predicted Low must not rise.

In [ ]:
for task, summary in report['task_summaries'].items():
    print('\n', task.upper())
    display(summary['fold_metrics'])
    if task == 'risk':
        print('Risk safety checks:', summary['risk_guardrails'])